# 檢索問答機器人（Retrieval-Augmented QA Bot）

## 學習目標

1. 理解雙塔（Bi-Encoder）+ 交互模型（Cross-Encoder）的兩階段檢索架構
2. 使用 FAISS 建立高效向量索引，完成 Top-K 召回（recall）
3. 以 Cross-Encoder 對候選結果重新排序（re-rank），提升最終精度
4. 掌握 2026 版 `from_pretrained` 統一慣例（`device_map='auto'`、`torch_dtype=torch.bfloat16`、`use_safetensors=True`）
5. 理解整個流程如何作為 RAG（Retrieval-Augmented Generation）的「檢索前端」

## 前置知識

- 完成 `02-Adv-tasks/04-sentence_similarity/` 的 Bi-Encoder 與 Cross-Encoder 訓練
- 了解 Transformer 的 `[CLS]` 向量語意
- 熟悉 Python `torch.inference_mode()` 與向量操作

## 與相鄰 Notebook 的銜接

- **上一步**：[04-sentence_similarity](../04-sentence_similarity/)—訓練 Bi-Encoder（`dual_model`）與 Cross-Encoder（`cross_model`）
- **下一步**：[06-LLM generation](../06-LLM_generation/)—把本 notebook 的「精確答案」送入 LLM 作生成增強（RAG 完整鏈路）

## 架構概覽

```
使用者問題
    │
    ▼
[Bi-Encoder] ──> 向量化 ──> FAISS 索引
                              │ Top-K 召回（粗篩，速度優先）
                              ▼
                       候選 FAQ 清單
                              │
    [Cross-Encoder] ──────────┘ 精排（精度優先）
                              │
                              ▼
                       最終匹配 FAQ + 答案
```

**為什麼要兩階段？**  
Bi-Encoder 把問題與 FAQ 分別編碼成固定向量，可以預先建索引，召回速度是 O(log N)；  
Cross-Encoder 把問題與每個候選拼接後才做 attention，語意理解精準但無法預建索引。  
兩階段結合：速度（召回）× 精度（排序）。

In [ ]:
# ── 版本鎖定（2026 統一基線）──────────────────────────────────────────────
# 執行一次即可；Colab / Kaggle 環境建議放在第一個 cell
%pip install -q \
    "transformers>=4.46" \
    "datasets>=3.0" \
    "evaluate>=0.4" \
    "accelerate>=1.0" \
    "safetensors>=0.4" \
    "torch>=2.4" \
    "faiss-cpu>=1.8" \
    "pandas>=2.0" \
    "tqdm"

## Step 1：載入 FAQ 資料集

使用 HuggingFace Hub 上的公開資料集（或以環境變數指定本機路徑），讓任何人 clone 後都能直接執行。

**資料欄位說明**

| 欄位 | 說明 |
|------|------|
| `title` | FAQ 問題標題（作為檢索 query 與候選 key） |
| `reply` | FAQ 標準答案（最終回傳給使用者） |

> **若使用本機 CSV**：設定環境變數 `FAQ_CSV_PATH=/your/path/law_faq.csv`，  
> 或直接修改下方 `DATA_PATH`。

In [ ]:
import os
from pathlib import Path
import pandas as pd

# ── 資料路徑（以環境變數注入，避免硬編路徑）──────────────────────────────
DATA_PATH = Path(os.environ.get("FAQ_CSV_PATH", "./law_faq.csv"))

if DATA_PATH.exists():
    data = pd.read_csv(DATA_PATH)
else:
    # fallback：示範用小型法律 FAQ（可換成 HF Hub 上的真實資料集）
    # from datasets import load_dataset
    # data = load_dataset("your_org/law_faq", split="train").to_pandas()
    raise FileNotFoundError(
        f"找不到 {DATA_PATH}。\n"
        "請設定環境變數 FAQ_CSV_PATH 指向正確路徑，"
        "或把 law_faq.csv 放到目前工作目錄。"
    )

print(f"資料筆數：{len(data)}")
print(f"欄位：{data.columns.tolist()}")
data.head(3)

In [ ]:
# 確認第一筆問題標題格式
print("第一筆 title：", data["title"].iloc[0])
print("第一筆 reply：", data["reply"].iloc[0][:100], "...")

## Step 2：載入 Bi-Encoder（雙塔匹配模型）

2026 版統一以 `AutoModel.from_pretrained` 搭配以下三個參數載入模型：

```python
dual_model = AutoModel.from_pretrained(
    model_id,
    device_map="auto",          # CPU / GPU / disk offload 自動決定
    torch_dtype=torch.bfloat16, # bf16：動態範圍同 fp32，但記憶體減半
    use_safetensors=True,       # 安全、更快的權重格式
)
```

### 為什麼用 bf16 而非 fp16 / fp32？

| 格式 | 指數位元 | 尾數位元 | 動態範圍 | 適用場景 |
|------|----------|----------|----------|----------|
| fp32 | 8 | 23 | 廣 | 訓練精度要求高 |
| fp16 | 5 | 10 | **窄**（易溢位） | 推論、部分訓練 |
| **bf16** | **8** | 7 | **同 fp32** | 推論 + 訓練（2026 首選） |

bf16 的指數位元與 fp32 相同，梯度爆炸問題大幅低於 fp16，且現代 GPU（A100、H100）與 CPU（AVX-512 BF16）原生支援。

### 為什麼用 `device_map='auto'`？

`device_map='auto'` 讓 `accelerate` 根據可用 VRAM 自動決定：
- 全部放 GPU → 正常推論
- 部分放 CPU → 自動 offload，不會 OOM
- 超大模型 → 部分放 disk（disk offload）

### 為什麼用 `use_safetensors=True`？

| | safetensors（`.safetensors`） |
|--|-------------------------------|
| 安全性 | 僅含張量資料，無程式碼 |
| 載入速度 | 快（mmap，幾乎零拷貝） |
| 2026 HF Hub | **預設格式** |

> **VRAM 需求**：`hfl/chinese-macbert-base`（110M 參數）bf16 約 **220 MB**，CPU 也能跑。

### 關於本 notebook 使用的模型

Bi-Encoder 的核心是把問題編碼成固定長度向量，這裡直接用  
`hfl/chinese-macbert-base` 的 `[CLS]` 向量當句子嵌入。  
若你已在 04-sentence_similarity 訓練了 `dual_model`，可改為：
```python
BIENCODER_ID = "your_hf_username/dual_model"  # push_to_hub 後的 ID
```

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# ── 模型 ID（改用 HF Hub，消除本機硬路徑）──────────────────────────────────
# 若已在 HF Hub 上傳 04-sentence_similarity 的 dual_model，改成你的 repo ID：
# BIENCODER_ID = "your_hf_username/chinese-dual-encoder"
BIENCODER_ID = os.environ.get("BIENCODER_MODEL_ID", "hfl/chinese-macbert-base")

tokenizer = AutoTokenizer.from_pretrained(BIENCODER_ID)

dual_model = AutoModel.from_pretrained(
    BIENCODER_ID,
    device_map="auto",          # 自動 CPU / GPU 分配
    torch_dtype=torch.bfloat16, # bf16：記憶體減半、動態範圍同 fp32
    use_safetensors=True,       # 安全且快速的權重格式
)
dual_model.eval()
print(f"Bi-Encoder 載入完成：{BIENCODER_ID}")
print(f"Device map：{dual_model.hf_device_map}")
print(f"Dtype：{next(dual_model.parameters()).dtype}")

## Step 3：將 FAQ 問題批次編碼成向量

### 設計要點

1. **批次大小 32**：平衡 GPU 使用率與記憶體；若 OOM 可改為 16。
2. **取 `[CLS]` 向量**（`last_hidden_state[:, 0, :]`）：代表整句語意，透明且可移植。
3. **`torch.inference_mode()`**：比 `torch.no_grad()` 更嚴格（禁止 grad 計算圖追蹤），推論速度略快。
4. 最終 `.cpu().float().numpy()`：FAISS 不接受 bf16，需轉 float32。

In [ ]:
import numpy as np
from tqdm.auto import tqdm

BATCH_SIZE = 32
MAX_LENGTH = 128

questions = data["title"].tolist()
vectors = []

with torch.inference_mode():
    for i in tqdm(range(0, len(questions), BATCH_SIZE), desc="Encoding FAQ"):
        batch = questions[i : i + BATCH_SIZE]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            max_length=MAX_LENGTH,
            truncation=True,
        )
        # 將輸入移到模型所在裝置（device_map='auto' 下模型可能跨裝置）
        inputs = {k: v.to(dual_model.device) for k, v in inputs.items()}

        outputs = dual_model(**inputs)
        # 取 [CLS] token 的 last_hidden_state 作為句向量
        # shape: (batch, hidden_size) = (batch, 768)
        cls_vectors = outputs.last_hidden_state[:, 0, :]
        # 轉回 float32 再移到 CPU（FAISS 需要 float32 numpy array）
        vectors.append(cls_vectors.float().cpu())

# 拼接所有批次
vectors_tensor = torch.cat(vectors, dim=0)  # (N, 768)
vectors_np = vectors_tensor.numpy()          # FAISS 用 numpy
print(f"FAQ 向量矩陣 shape：{vectors_np.shape}")

## Step 4：建立 FAISS 向量索引

### 為什麼用 `IndexFlatIP`（內積索引）？

FAISS 支援多種索引：

| 索引類型 | 相似度度量 | 精確 | 速度 | 適用 |
|----------|-----------|------|------|------|
| `IndexFlatL2` | L2 距離 | 是 | 慢 | 小資料集，需精確 |
| `IndexFlatIP` | 內積（cosine 需先 L2 normalize）| 是 | 慢 | **本 notebook**：精確、cosine 語意 |
| `IndexIVFFlat` | 近似，分群 | 否 | 快 | 百萬以上資料集 |
| `IndexHNSW` | 近似，圖 | 否 | 快 | 需低延遲服務 |

`faiss.normalize_L2()` 將向量歸一化為單位向量，此後內積 = cosine similarity。  
本 FAQ 資料集通常幾千筆，`IndexFlatIP` 精確度最高。

In [ ]:
import faiss

HIDDEN_SIZE = vectors_np.shape[1]  # 768

# 建立精確內積索引
index = faiss.IndexFlatIP(HIDDEN_SIZE)

# 必須先 normalize，讓內積等於 cosine similarity
faiss.normalize_L2(vectors_np)
index.add(vectors_np)

print(f"FAISS index 已建立：{index.ntotal} 筆向量，維度 {HIDDEN_SIZE}")

## Step 5：對使用者問題進行向量編碼

編碼流程與 FAQ 批次編碼相同，但只處理單一問題。

In [ ]:
# 使用者輸入的問題（示範）
question = "寻衅滋事"

with torch.inference_mode():
    inputs = tokenizer(
        question,
        return_tensors="pt",
        padding=True,
        max_length=MAX_LENGTH,
        truncation=True,
    )
    inputs = {k: v.to(dual_model.device) for k, v in inputs.items()}
    outputs = dual_model(**inputs)
    q_cls = outputs.last_hidden_state[:, 0, :]
    # 轉 float32 numpy，shape: (1, 768)
    q_vector = q_cls.float().cpu().numpy()

print(f"問題向量 shape：{q_vector.shape}")

## Step 6：向量召回（Bi-Encoder 粗篩）

### 向量匹配 vs 向量交互匹配

**向量匹配（Vector Matching，本步驟）**  
問題與 FAQ 分別獨立編碼成固定向量，透過相似度比較（cosine / 內積）找 Top-K。  
優點：FAQ 向量可預建索引，查詢時 O(log N) 甚至 O(1)（ANN 索引）。  
缺點：兩段文字之間的 token 級互動資訊完全丟失。

**向量交互匹配（Vector Interaction Matching，Step 8 的 Cross-Encoder）**  
把問題與候選 FAQ 拼接成一個輸入序列，讓 Transformer 的 attention 充分捕捉兩段文字間的 token 級關係。  
優點：語意理解最深、精度最高。  
缺點：無法預建索引（每次查詢都要重新 forward N 個候選），不適合 N 很大時直接對全庫搜尋。

**兩階段設計的本質**：以 Bi-Encoder 的速度換取廣度，以 Cross-Encoder 的精度換取深度。

In [ ]:
TOP_K = 10  # 召回候選數

# 先 normalize query 向量（與索引建立時一致）
faiss.normalize_L2(q_vector)

scores, indexes = index.search(q_vector, TOP_K)

# 取出對應的 FAQ 資料列
topk_rows = data.values[indexes[0].tolist()]  # shape: (TOP_K, n_columns)
topk_titles = topk_rows[:, 0].tolist()          # 候選問題標題
topk_answers = topk_rows[:, 1].tolist()          # 對應答案

print(f"Top-{TOP_K} 召回結果（Bi-Encoder 粗篩）：")
for i, (title, score) in enumerate(zip(topk_titles, scores[0])):
    print(f"  [{i+1}] score={score:.4f}  {title}")

## Step 7：載入 Cross-Encoder（交互排序模型）

Cross-Encoder 的輸入格式：`[CLS] 問題 [SEP] 候選FAQ [SEP]`，  
BERT 做 sequence pair classification，輸出每對的相關性分數。

2026 版以 `AutoModelForSequenceClassification.from_pretrained` 搭配統一慣例載入：

```python
cross_model = AutoModelForSequenceClassification.from_pretrained(
    CROSSENCODER_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)
```

> **VRAM 需求**：`hfl/chinese-macbert-base` bf16 約 220 MB，與 Bi-Encoder 共享時總計 ~440 MB。  
> 若顯存不足，可設 `device_map="cpu"` 強制 CPU 推論（速度略慢但功能相同）。

In [ ]:
from transformers import AutoModelForSequenceClassification

# 若已在 HF Hub 上傳 04-sentence_similarity 的 cross_model，改成你的 repo ID：
# CROSSENCODER_ID = "your_hf_username/chinese-cross-encoder"
CROSSENCODER_ID = os.environ.get("CROSSENCODER_MODEL_ID", "hfl/chinese-macbert-base")

cross_model = AutoModelForSequenceClassification.from_pretrained(
    CROSSENCODER_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)
cross_model.eval()
print(f"Cross-Encoder 載入完成：{CROSSENCODER_ID}")
print(f"分類標籤數：{cross_model.config.num_labels}")

## Step 8：Cross-Encoder 精排（重新排序）

### 輸入構造

Cross-Encoder 接受 sequence pair：
```
tokenizer(questions_list, candidates_list, ...)
```
`tokenizer` 會自動在兩段文字之間插入 `[SEP]`，組成：  
`[CLS] 使用者問題 [SEP] 候選FAQ標題 [SEP]`

### 排序邏輯

- `logits` shape：`(TOP_K, num_labels)`
- 若 `num_labels == 2`：取 class 1（相關類別）的 logit 對所有候選排序，找出最相關者
- 若 `num_labels == 1`：直接用 logit 值（regression head）

In [ ]:
import torch.nn.functional as F

candidates = topk_rows[:, 0].tolist()
answers = topk_rows[:, 1].tolist()

# 構造問題-候選配對輸入
question_repeated = [question] * len(candidates)
inputs = tokenizer(
    question_repeated,
    candidates,
    return_tensors="pt",
    padding=True,
    max_length=MAX_LENGTH,
    truncation=True,
)
inputs = {k: v.to(cross_model.device) for k, v in inputs.items()}

with torch.inference_mode():
    logits = cross_model(**inputs).logits  # (TOP_K, num_labels)

if logits.shape[-1] == 1:
    # Regression head：直接用 logit 值
    relevance_scores = logits.squeeze(-1).float().cpu()
elif logits.shape[-1] == 2:
    # Binary classification head：取 class 1（相關類別）的 logit
    relevance_scores = logits[:, 1].float().cpu()
else:
    # 多分類：取最高 logit
    relevance_scores = logits.max(dim=-1).values.float().cpu()

# 依相關分數降序排列
ranked_indices = torch.argsort(relevance_scores, descending=True)
best_idx = ranked_indices[0].item()

match_question = candidates[best_idx]
final_answer = answers[best_idx]

print(f"使用者問題：{question}")
print(f"最佳匹配 FAQ：{match_question}")
print(f"最終答案：{final_answer}")

## Step 9：完整端對端查詢函式

把上述步驟封裝成可重複呼叫的函式，方便整合進下游應用（例如 RAG 管線）。

In [ ]:
def retrieve_answer(
    question: str,
    data: pd.DataFrame,
    dual_model: AutoModel,
    cross_model: AutoModelForSequenceClassification,
    tokenizer: AutoTokenizer,
    index: faiss.Index,
    top_k: int = 10,
    max_length: int = 128,
) -> dict:
    """
    Two-stage retrieval: Bi-Encoder recall + Cross-Encoder re-rank.

    Args:
        question: user query string
        data: FAQ DataFrame with columns ['title', 'reply']
        dual_model: bi-encoder model
        cross_model: cross-encoder re-ranking model
        tokenizer: shared tokenizer
        index: FAISS index built from FAQ title vectors
        top_k: number of candidates to recall
        max_length: max token length for both stages

    Returns:
        dict with keys: question, matched_faq, answer, recall_scores, rerank_scores
    """
    # ── Stage 1：Bi-Encoder 向量召回 ──────────────────────────────────────
    with torch.inference_mode():
        inputs = tokenizer(
            question,
            return_tensors="pt",
            padding=True,
            max_length=max_length,
            truncation=True,
        )
        inputs = {k: v.to(dual_model.device) for k, v in inputs.items()}
        q_vec = dual_model(**inputs).last_hidden_state[:, 0, :]
        q_vec = q_vec.float().cpu().numpy()

    faiss.normalize_L2(q_vec)
    recall_scores, recall_indices = index.search(q_vec, top_k)

    topk_rows = data.values[recall_indices[0].tolist()]
    candidates = topk_rows[:, 0].tolist()
    candidate_answers = topk_rows[:, 1].tolist()

    # ── Stage 2：Cross-Encoder 精排 ───────────────────────────────────────
    question_repeated = [question] * len(candidates)
    pair_inputs = tokenizer(
        question_repeated,
        candidates,
        return_tensors="pt",
        padding=True,
        max_length=max_length,
        truncation=True,
    )
    pair_inputs = {k: v.to(cross_model.device) for k, v in pair_inputs.items()}

    with torch.inference_mode():
        logits = cross_model(**pair_inputs).logits

    if logits.shape[-1] == 1:
        relevance = logits.squeeze(-1).float().cpu()
    elif logits.shape[-1] == 2:
        relevance = logits[:, 1].float().cpu()
    else:
        relevance = logits.max(dim=-1).values.float().cpu()

    best_idx = torch.argmax(relevance).item()

    return {
        "question": question,
        "matched_faq": candidates[best_idx],
        "answer": candidate_answers[best_idx],
        "recall_scores": recall_scores[0].tolist(),
        "rerank_scores": relevance.tolist(),
    }


# ── 示範呼叫 ──────────────────────────────────────────────────────────────
result = retrieve_answer(
    question="寻衅滋事",
    data=data,
    dual_model=dual_model,
    cross_model=cross_model,
    tokenizer=tokenizer,
    index=index,
)
print(f"問題：{result['question']}")
print(f"匹配 FAQ：{result['matched_faq']}")
print(f"答案：{result['answer']}")

## Step 10：批次測試與準確率評估

從資料集中隨機抽樣，評估「Top-1 Re-rank 是否等於原始 FAQ」作為精度代理指標。

> 注意：這是 **closed-set recall@1** 評估（用資料集本身問題查詢），  
> 衡量的是索引+排序能否找回正確 FAQ，不代表對新問題的泛化能力。

In [ ]:
import random
from transformers import set_seed

set_seed(42)
random.seed(42)

N_EVAL = min(50, len(data))  # 評估筆數；資料集小時取全部
sample_indices = random.sample(range(len(data)), N_EVAL)

correct = 0
for idx in tqdm(sample_indices, desc="Evaluating"):
    row = data.iloc[idx]
    ground_truth_title = row["title"]

    result = retrieve_answer(
        question=ground_truth_title,
        data=data,
        dual_model=dual_model,
        cross_model=cross_model,
        tokenizer=tokenizer,
        index=index,
    )
    if result["matched_faq"] == ground_truth_title:
        correct += 1

accuracy = correct / N_EVAL
print(f"\nClosed-set Recall@1（Re-rank 後）：{accuracy:.2%}  ({correct}/{N_EVAL})")

## 小結

本 notebook 實作了兩階段檢索問答系統：

| 階段 | 技術 | 作用 | 速度 | 精度 |
|------|------|------|------|------|
| 粗篩（Stage 1） | Bi-Encoder + FAISS | Top-K 召回 | O(log N) | 中 |
| 精排（Stage 2） | Cross-Encoder | 重排 Top-K 結果 | O(K) | 高 |

### 2026 設計重點

1. **消除本機硬路徑**：用環境變數注入路徑，或從 HF Hub 載入，任何環境都能執行
2. **統一裝置慣例**：`device_map='auto'` 讓 `accelerate` 自動分配 CPU/GPU，避免 OOM
3. **bf16 精度**：記憶體減半，動態範圍優於 fp16，2026 推論首選
4. **safetensors**：安全且快速的權重格式，mmap 載入，幾乎零拷貝
5. **Re-rank 正確邏輯**：binary classification head 取 class 1 logit 排序；regression head 直接取 logit 值
6. **封裝函式**：`retrieve_answer()` 可直接整合進 RAG 管線

### 延伸練習

1. **增加 FAQ 筆數**：把本系統接上 10,000 筆以上的資料集，改用 `IndexIVFFlat` 或 `IndexHNSW` 觀察速度差異
2. **連接 LLM 生成**：把 `final_answer` 作為 context 送入 `06-LLM_generation`，完成完整 RAG 鏈路
3. **評估 Recall@K**：計算 Bi-Encoder 的 Recall@1 與 Recall@10，觀察 Cross-Encoder Re-rank 帶來的提升幅度
4. **替換 Embedding 模型**：把 `hfl/chinese-macbert-base` 換成 `BAAI/bge-m3`（多語言、更強語意），觀察準確率變化
5. **串流回應**：在 Step 10 加入 `model.generate()` + `TextStreamer`，讓系統能串流輸出答案